# Core Python Concepts for Agent Development

### **Lists & Tuples**

- List `[]`: Mutable sequences. Use for memory buffers or task queues.

- Tuple `()`: Immutable sequences. Use for fixed configurations or coordinates.

In [1]:
# Use Case: Storing message history
history = ["Hello!", "How can I help?"] 
history.append("What is the weather?")

# Use Case: API Credentials (Fixed)
api_config = ("openai_v3", "https://api.openai.com/v1")

### **Strings & Formatting**
The bread and butter of agents. Used for Prompt Engineering.

In [2]:
agent_role = "Research Assistant"
user_query = "Explain Quantum Physics."

# f-strings are essential for dynamic prompting
prompt = f"You are a {agent_role}. Answer this: {user_query}"
print(prompt.upper())

YOU ARE A RESEARCH ASSISTANT. ANSWER THIS: EXPLAIN QUANTUM PHYSICS.


### **Conditionals & Loops**

Used for **Agent Logic** (deciding which tool to use) and **Iteration** (retrying failed tasks).

In [3]:
# Use Case: Router logic
status_code = 200
if status_code == 200:
    print("Proceeding to next task...")
else:
    print("Retrying...")

# Use Case: Processing a list of sub-tasks
tasks = ["Search web", "Summarize", "Email user"]
for task in tasks:
    print(f"Executing: {task}")

Proceeding to next task...
Executing: Search web
Executing: Summarize
Executing: Email user


### **Decorators**
Used for **Logging** or **Timing** agent thoughts without cluttering the main logic.

In [4]:
def log_action(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] Agent is calling: {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_action
def think():
    return "I should check the database."

### **OOP (Object Oriented Programming)**
Essential for maintaining **Agent State** (memory, personality, and API keys).

In [5]:
class BaseAgent:
    def __init__(self, name):
        self.name = name
        self.memory = []

    def add_to_memory(self, text):
        self.memory.append(text)

## Custom Autonomous Agent Building

Using Python only, we building a functional **Autonomous Agent**, we need to introduce **Tools**. An agent uses tools (functions) to interact with the outside world, such as searching the web, calculating math, or reading files.

#### **1. Project Structure & Security**
For a professional setup, we separate logic into different files (modules).

- `.env`: Stores your secret OPENAI_API_KEY.

- `requirements.txt`: Lists dependencies (e.g., `openai`, `python-dotenv`).

- `tools.py`: A module containing functions the agent can call.

- `agent.py`: The main logic.

**Environment Variables (.env)**
Never hardcode keys. Create a file named `.env`:
```env
OPENAI_API_KEY=your_openai_api_key_here
```

#### **2. Creating the Tools Module (`tools.py`)**
This demonstrates **Modules** and **Functional Logic**. We define a set of tools that our agent can "choose" to use.

In [ ]:
# tools.py
import datetime
import math
import random

def get_current_time():
    """Returns the current date and time."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
    """Evaluates a mathematical expression string. e.g., '50 * 2 + 10'"""
    try:
        # Using eval safely by limiting globals (Basic logic)
        result = eval(expression, {"__builtins__": None}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error calculating: {e}"

def get_weather(city):
    """Simulates fetching weather data for a specific city."""
    # In a real scenario, you'd use a weather API here
    weathers = ["Sunny", "Cloudy", "Rainy", "Snowing", "Windy"]
    temp = random.randint(15, 35)
    return f"The weather in {city} is {random.choice(weathers)} with a temperature of {temp}°C."

def convert_currency(amount, from_currency, to_currency):
    """Simulates currency conversion. Args: amount, from_code, to_code."""
    # Mock rates
    rates = {"USD_to_EUR": 0.92, "USD_to_GBP": 0.79, "USD_to_INR": 83.0}
    key = f"{from_currency.upper()}_to_{to_currency.upper()}"
    
    if key in rates:
        converted = float(amount) * rates[key]
        return f"{amount} {from_currency} is approximately {converted:.2f} {to_currency}."
    else:
        return "Error: Exchange rate not available for this pair."

def generate_password(length=12):
    """Generates a random secure password string."""
    chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789!@#$%^&*"
    password = "".join(random.choice(chars) for _ in range(int(length)))
    return f"Generated Password: {password}"

# Update this dictionary so the Agent can see the new tools.
available_tools = {
    "get_current_time": get_current_time,
    "calculator": calculator,
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "generate_password": generate_password
}

#### **3. The Advanced Agent Implementation**
We will use `python-dotenv` to load the key and **OOP** to build the engine.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from tools import available_tools  # Ensure your tools.py is updated with the 5 tools

# Load Environment Variables
load_dotenv()

class AutonomousAgent:
    def __init__(self, model="gpt-4"):
        self.api_key = os.getenv("OPENAI_API_KEY")
        self.model = model
        self.url = "https://api.openai.com/v1/chat/completions"
        
        # Enhanced System Prompt to prevent tool name/argument errors
        self.system_prompt = (
            "You are a helpful AI Agent with access to tools. "
            "If you need to use a tool, you MUST respond in JSON format: "
            '{"tool": "function_name", "args": [arg1, arg2]}. '
            f"Available tools: {list(available_tools.keys())}. "
            "Notes: 'convert_currency' takes [amount, from_code, to_code]. "
            "If no tool is needed, respond with plain text."
        )
        
        self.memory = [{"role": "system", "content": self.system_prompt}]

    def _call_llm(self, message, role="user"):
        """Internal method to talk to OpenAI."""
        self.memory.append({"role": role, "content": message})
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": self.model,
            "messages": self.memory,
            "temperature": 0  # Low temperature for consistent tool calling
        }

        try:
            response = requests.post(self.url, headers=headers, json=payload)
            response.raise_for_status()
            content = response.json()['choices'][0]['message']['content']
            
            # Store assistant response in memory
            self.memory.append({"role": "assistant", "content": content})
            return content
        except Exception as e:
            return f"Error: {e}"

    def run(self, task):
        """The main loop: Think -> Act -> Observe -> Final Response."""
        # 1. THINK: Send user task to LLM
        llm_response = self._call_llm(task)
        
        # 2. ACT: Check if the response is a JSON tool call
        try:
            # We check if the response looks like JSON
            if '{"tool"' in llm_response.replace(" ", ""):
                # Clean the response string in case LLM added markdown triple backticks
                clean_json = llm_response.replace("```json", "").replace("```", "").strip()
                action = json.loads(clean_json)
                
                tool_name = action['tool']
                args = action.get('args', [])

                # Validate tool existence
                if tool_name in available_tools:
                    print(f"[*] Executing Tool: {tool_name} with {args}")
                    
                    # Execute the tool from tools.py
                    observation = available_tools[tool_name](*args)
                    print(f"[*] Observation: {observation}")

                    # 3. OBSERVE: Send tool results back to LLM for a final summary
                    final_reply = self._call_llm(
                        f"The tool '{tool_name}' returned: {observation}. "
                        "Based on this, give the user a clear final answer.",
                        role="user"
                    )
                    return final_reply
                else:
                    return f"Agent tried to use unknown tool: {tool_name}"

        except (json.JSONDecodeError, TypeError) as e:
            # If JSON is malformed or tool arguments are wrong, return the raw response
            return llm_response

        # If it wasn't a tool call, just return the text
        return llm_response

# --- Interactive Session ---
if __name__ == "__main__":
    bot = AutonomousAgent()
    print("--- Agent Online (Type 'exit' to stop) ---")
    
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit", "quit", "bye"]:
            break
        
        answer = bot.run(user_input)
        print(f"\nAgent: {answer}\n")